In [ ]:
import pandas as pd
import re
import os
import xml.etree.ElementTree as ET
from urllib.request import urlopen
from tqdm import trange

# 디렉토리 생성(없으면)
output_dir = '법령별 csv 저장(현역)'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# CSV 파일 로드
law_list = pd.read_csv("전체_법률_목록(2026.03.16).csv")  # 파일명 변경

def remove_tag(content):
    """XML에서 사용된 특수문자 제거용 함수"""
    return re.sub('<.*?>', '', content)

### {mst_value}를 바꿔가면서 사용할 베이스 url
### OC 파트 반드시 확인!
url_template = "https://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=law&MST={mst_value}&type=XML"


# 세션 제한(~1500건)으로 인해 0~1499번까지만 처리
for i in trange(1500):
    mst_value = law_list.loc[i, '법령MST']  # MST 값: 각 법령을 식별하는 고유 ID
    법령명 = law_list.loc[i, '법령명'].strip()  # 법령명: 법령의 이름
    url = url_template.format(mst_value=mst_value)  # MST 값을 사용하여 URL 생성
    
    try:
        # URL에서 XML 데이터 요청 및 읽기
        response = urlopen(url).read()  # URL로부터 응답 데이터 읽기
        response = response.decode('utf-8', errors='ignore')  # UTF-8로 디코딩하여 문자열로 변환
        response = re.sub(r'<script.*?/>', '', response)  # <script/> 태그 제거 (불필요한 내용 정리)
        
        # 응답이 XML 형식인지 확인
        if response.strip().startswith('<?xml'):
            root = ET.fromstring(response)  # XML 파싱하여 ElementTree 객체로 변환
        else:
            print(f"Non-XML response received for {법령명}")
            continue

        all_rows = []  # 수집된 모든 법령 데이터를 저장할 리스트 초기화

        # XML에서 <조문단위> 태그를 모두 찾아서 반복 처리
        for clause in root.findall(".//조문단위"):
            조문제목 = clause.find('조문제목')

            # 기본 조문 정보 수집
            조문정보 = {
                '법령명': 법령명,
                '조문번호': clause.find('조문번호').text if clause.find('조문번호') is not None else '',
                '조문제목': remove_tag(조문제목.text) if 조문제목 is not None else '',
                '조문내용': remove_tag(clause.find('조문내용').text) if clause.find('조문내용') is not None else '',
                '조문시행일자': clause.find('조문시행일자').text if clause.find('조문시행일자') is not None else '',
                '조문변경여부': clause.find('조문변경여부').text if clause.find('조문변경여부') is not None else ''
            }

            if 조문정보['조문내용']:
                all_rows.append({**조문정보, '항': '', '호': '', '목': ''})

            for depth1 in clause.findall('항'):
                항내용 = depth1.find('항내용').text.strip() if depth1.find('항내용') is not None else ''
                항_row = {**조문정보, '항': 항내용, '호': '', '목': ''}
                all_rows.append(항_row)

                for depth2 in depth1.findall('호'):
                    호내용 = depth2.find('호내용').text.strip() if depth2.find('호내용') is not None else ''
                    호_row = {**항_row, '호': 호내용, '목': ''}  # 호내용에 이미 번호 포함
                    all_rows.append(호_row)

                    for depth3 in depth2.findall('목'):
                        목내용 = depth3.find('목내용').text.strip() if depth3.find('목내용') is not None else ''
                        목_row = {**호_row, '목': 목내용}
                        all_rows.append(목_row)

        if all_rows:
            file_path = os.path.join(output_dir, f"{법령명}.csv")
            pd.DataFrame(all_rows).to_csv(file_path, index=False, encoding='utf-8-sig', sep='\t')
            print(f"Saved {법령명} to {output_dir}")
        else:
            print(f"No data found for {법령명}")

    except Exception as e:
        # break → continue 로 변경: 에러 발생 시 해당 법령만 건너뛰고 계속 진행
        print(f"Error fetching or parsing XML for {법령명} (MST: {mst_value}): {e}")
        continue

### 1500번째 행부터 리트라이 (세션 갱신 후 실행)

In [ ]:
import pandas as pd
import re
import os
import xml.etree.ElementTree as ET
from urllib.request import urlopen
from tqdm import trange

output_dir = '법령별 csv 저장(현역)'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

law_list = pd.read_csv("전체_법률_목록(2026.03.16).csv")  # 파일명 변경

def remove_tag(content):
    return re.sub('<.*?>', '', content)

url_template = "https://www.law.go.kr/DRF/lawService.do?OC=silverbreak&target=law&MST={mst_value}&type=XML"


# 1500번 인덱스부터 끝까지 처리 (Cell 1에서 0~1499 처리했으므로)
for i in trange(1500, len(law_list)):
    mst_value = law_list.loc[i, '법령MST']
    법령명 = law_list.loc[i, '법령명'].strip()
    url = url_template.format(mst_value=mst_value)
    
    try:
        response = urlopen(url).read()
        response = response.decode('utf-8', errors='ignore')
        response = re.sub(r'<script.*?/>', '', response)
        
        if response.strip().startswith('<?xml'):
            root = ET.fromstring(response)
        else:
            print(f"Non-XML response received for {법령명}")
            continue

        all_rows = []

        for clause in root.findall(".//조문단위"):
            조문제목 = clause.find('조문제목')

            조문정보 = {
                '법령명': 법령명,
                '조문번호': clause.find('조문번호').text if clause.find('조문번호') is not None else '',
                '조문제목': remove_tag(조문제목.text) if 조문제목 is not None else '',
                '조문내용': remove_tag(clause.find('조문내용').text) if clause.find('조문내용') is not None else '',
                '조문시행일자': clause.find('조문시행일자').text if clause.find('조문시행일자') is not None else '',
                '조문변경여부': clause.find('조문변경여부').text if clause.find('조문변경여부') is not None else ''
            }

            if 조문정보['조문내용']:
                all_rows.append({**조문정보, '항': '', '호': '', '목': ''})

            for depth1 in clause.findall('항'):
                항내용 = depth1.find('항내용').text.strip() if depth1.find('항내용') is not None else ''
                항_row = {**조문정보, '항': 항내용, '호': '', '목': ''}
                all_rows.append(항_row)

                for depth2 in depth1.findall('호'):
                    호내용 = depth2.find('호내용').text.strip() if depth2.find('호내용') is not None else ''
                    호_row = {**항_row, '호': 호내용, '목': ''}  # 호내용에 이미 번호 포함
                    all_rows.append(호_row)

                    for depth3 in depth2.findall('목'):
                        목내용 = depth3.find('목내용').text.strip() if depth3.find('목내용') is not None else ''
                        목_row = {**호_row, '목': 목내용}
                        all_rows.append(목_row)

        if all_rows:
            file_path = os.path.join(output_dir, f"{법령명}.csv")
            pd.DataFrame(all_rows).to_csv(file_path, index=False, encoding='utf-8-sig', sep='\t')
            print(f"Saved {법령명} to {output_dir}")
        else:
            print(f"No data found for {법령명}")

    except Exception as e:
        # break → continue 로 변경
        print(f"Error fetching or parsing XML for {법령명} (MST: {mst_value}): {e}")
        continue

### 중복 제거

In [ ]:
import pandas as pd
import os

def process_files(folder_path, target_folder):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
    
    for file_name in os.listdir(folder_path):
        try:
            if file_name.endswith('.csv'):
                file_path = os.path.join(folder_path, file_name)
                df = pd.read_csv(file_path, encoding='utf-8-sig', sep='\t')
                print(f"Processing : {file_path}")
                df = remove_duplicate_cell_contents(df, ['조문내용', '항', '호', '목'])
                df = df.fillna('')
                target_file_path = os.path.join(target_folder, file_name)
                df.to_csv(target_file_path, index=False, encoding='utf-8-sig', sep='\t')
                print(f"Updated file saved to: {target_file_path}")
        except Exception as e:
            print(f"Error processing file {file_name}: {e}")
            continue

def remove_duplicate_cell_contents(df, columns):
    for column in columns:
        last_seen = None
        for idx in df.index:
            current_value = df.at[idx, column]
            if current_value == last_seen:
                df.at[idx, column] = ''
            else:
                last_seen = current_value
    return df

folder_path = '법령별 csv 저장(현역)'
target_folder = '법령별 csv 저장(현역) 중복제거'
process_files(folder_path, target_folder)

### 텍스트 통합

In [2]:
import os
import pandas as pd
from tqdm import tqdm

source_dir = '법령 csv 저장(26.03.16 기준) 중복제거'
output_dir = '법령_통합텍스트(26.03.16)'
columns_to_combine = ['조문내용', '항', '호', '목']

os.makedirs(output_dir, exist_ok=True)

print(f"'{source_dir}' 폴더의 CSV 파일들을 읽어 '{output_dir}' 폴더에 TXT 파일로 통합합니다.")
print(f"통합 대상 컬럼: {columns_to_combine}")

if not os.path.isdir(source_dir):
    print(f"오류: 원본 폴더 '{source_dir}'를 찾을 수 없습니다. 경로를 확인해주세요.")
else:
    for filename in tqdm(os.listdir(source_dir)):
        if filename.endswith('.csv'):
            try:
                source_path = os.path.join(source_dir, filename)
                df = pd.read_csv(source_path, sep='\t')
                existing_columns = [col for col in columns_to_combine if col in df.columns]
                df[existing_columns] = df[existing_columns].fillna('').astype(str)
                combined_series = df[existing_columns].apply(lambda row: ' '.join(row.values), axis=1)
                full_text = '\n'.join(combined_series)
                output_filename = filename.replace('.csv', '.txt')
                output_path = os.path.join(output_dir, output_filename)
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(full_text)
            except Exception as e:
                print(f"오류 발생: {filename} 처리 중 문제 발생 - {e}")

    print(f"\n작업 완료! 모든 법령이 '{output_dir}' 폴더에 .txt 파일로 저장되었습니다.")

'법령 csv 저장(26.03.16 기준) 중복제거' 폴더의 CSV 파일들을 읽어 '법령_통합텍스트(26.03.16)' 폴더에 TXT 파일로 통합합니다.
통합 대상 컬럼: ['조문내용', '항', '호', '목']


100%|██████████| 1706/1706 [00:13<00:00, 127.64it/s]


작업 완료! 모든 법령이 '법령_통합텍스트(26.03.16)' 폴더에 .txt 파일로 저장되었습니다.
